# Provenance Management

This notebook shows how to upload provenance documents and related artifacts to the yProvStore service using the EOSC AAI.

In the first part of the notebook, a new provenance document is built from scratch, including all the core elements specified by the W3C PROV standard. The document reflects the workflow described in the user story.

In [ ]:
%pip install networkx

In [ ]:
from prov.model import ProvDocument, Namespace, PROV
doc = ProvDocument()

### Namespaces

In [ ]:
CMIP = Namespace("cmip",'https://www.wcrp-climate.org/wgcm-cmip')
YPROV = Namespace("yprov", "https://github.com/HPCI-Lab/yProvStore/")
EOSC = Namespace("eosc", "https://eosc.eu/")
LW = Namespace("lifewatch", "https://www.lifewatch.eu/")
ENES = Namespace("enes", "https://portal.enes.org/")
MATPLOTLIB = Namespace("matplotlib", "https://matplotlib.org/")

doc.add_namespace(CMIP)
doc.add_namespace(MATPLOTLIB)
doc.add_namespace(EOSC)
doc.add_namespace(LW)
doc.add_namespace(ENES)
doc.add_namespace(YPROV)

### Entities

In [ ]:
org_eosc = doc.agent(
    EOSC["EOSC"], {
        PROV["type"]: PROV["Organization"]
    }
)
org_lw = doc.agent(
    LW["LifeWatch"], {
        PROV["type"]: PROV["Organization"]
    }
)
org_enes = doc.agent(
    ENES["ENES"], {
        PROV["type"]: PROV["Organization"]
    }
)

eosc = doc.entity(
    EOSC["eosc_discovery_hub"],
    { 
        PROV["location"]: "https://search.userspace.sandbox.eosc-beyond.eu/search/all_collection?q=*"
    }
)

lw_catalog = doc.entity(
    LW["lifewatch_catalogue"],
    {
        PROV["type"]: PROV["Collection"],
        PROV["location"]: "https://metadatacatalogue.lifewatch.eu/srv/eng/catalog.search#/home"
    }
)

lw_record = doc.entity(
    LW["amphipod_species"],
    {
        PROV["type"]: PROV["Collection"],
        PROV["location"]: "https://metadatacatalogue.lifewatch.eu/srv/api/records/ac2350c3-b912-4ce6-89bb-c922e15ec2cc",
        PROV["label"]: "Amphipod species in European marine waters"
    }
)

gbif = doc.entity(
    LW["amphipod_species_distribution"], {
        PROV["location"]: "https://www.gbif.org/dataset/1260320d-5eb6-475b-bf4a-a60cf3f9e3e1",
        PROV["label"]: "GBIF distribution for amphipod species in European marine waters"
    }                 
) 

csv = doc.entity(
    LW["amphipod_species_data"], {
        PROV["type"]: "csv",
        PROV["label"]: "CSV file for amphipod species in European marine waters",
        PROV["location"]: "/home/jovyan/work/amphipod_species_EU_marine_waters.csv"
    }
)   

lw_df = doc.entity(
    LW["amphipod_species_df"], {
        PROV["type"]: "Pandas DataFrame",
        PROV["label"]: "Pandas DataFrame for amphipod species in European marine waters"
    }
)

lw_df_eu = doc.entity(
    LW["amphipod_species_df_eu"], {
        PROV["type"]: "Pandas DataFrame",
        PROV["label"]: "Subset over the EU domain"
    }
)

enes_catalog = doc.entity(
    ENES["enes_catalog"],{
        PROV["type"]: PROV["Collection"],
        PROV["location"]: "https://catalogue.eneslab.pilot.eosc-beyond.eu/"
    }
)

enes_dataset = doc.entity(
    CMIP["tas_historical"], {
        PROV["type"]: "Zarr",
        PROV["location"]: "https://object-store.brno.openstack.cloud.e-infra.cz/swift/v1/KEY_db4f6691a81f49d0a8dda59b26259b5b/vo.enes.org-cmip6/CMIP6.CMIP.CMCC.CMCC-ESM2.historical.r1i1p1f1.day.tas.gn/",
        PROV["label"]: "CMIP6 dataset for tas",
        CMIP["variable"]: "tas",
        CMIP["model"] : "CMCC-ESM2",
        CMIP["experiment"] : "historical",
        CMIP["frequency"] : "day"
    }
)

enes_xarray = doc.entity(
    CMIP["tas_historical_xarray"], {
        PROV["type"]: "Xarray DataArray",
        PROV["label"]: "CMIP6 dataset for tas",
        CMIP["variable"]: "tas",
        CMIP["model"] : "CMCC-ESM2",
        CMIP["experiment"] : "historical",
        CMIP["frequency"] : "day"
    }
)

enes_subset = doc.entity(
    CMIP["tas_historical_xarray_eu"], {
        PROV["label"]: "Subset over the EU domain",
        PROV["type"]: "Xarray DataArray",
        CMIP["variable"]: "tas",
        CMIP["model"] : "CMCC-ESM2",
        CMIP["experiment"] : "historical",
        CMIP["frequency"] : "day"
    }
)
    
output = doc.entity(
    MATPLOTLIB["map"], {
        PROV["label"]: "Temperature and species occurrences - Year 2000",
        PROV["type"]: "Matplotlib map",
        PROV["location"]: "/home/jovyan/work/temperature_and_species_occurrences_2000.png"
      }  
)

### Activities

In [ ]:
download_to_csv = doc.activity(LW["downloadToCSV"])

read_csv = doc.activity(LW["readCSV"])

subset_df = doc.activity(
    LW["spatialLWSubset"],
    other_attributes={
        PROV["label"]: "Spatial and temporal subset on biodiversity dataset"
    }
)

read_zarr = doc.activity(ENES["readZarr"])

subset_zarr = doc.activity(
    ENES["spatialClimateSubset"],
    other_attributes={
        PROV["label"]: "Spatial and temporal subset on climate dataset"
    }
)

plot = doc.activity(
    MATPLOTLIB["plotData"],
    other_attributes={
        PROV["label"]: "Data visualization"
    }
)

### Membership

In [ ]:
doc.wasAttributedTo(eosc, org_eosc)
doc.wasAttributedTo(lw_catalog, org_lw)
doc.wasAttributedTo(enes_catalog, org_enes)

doc.hadMember(eosc,lw_catalog)
doc.hadMember(eosc,enes_catalog)

doc.hadMember(lw_catalog,lw_record)
doc.hadMember(enes_catalog,enes_dataset)

### Usage, Generation and Derivation

In [ ]:
doc.used(download_to_csv, gbif)
doc.wasGeneratedBy(csv, download_to_csv)

doc.used(read_csv, csv)
doc.wasGeneratedBy(lw_df, read_csv)

doc.used(subset_df, lw_df)
doc.wasGeneratedBy(lw_df_eu, subset_df)
doc.wasDerivedFrom(lw_df_eu, lw_df)

doc.used(read_zarr, enes_dataset)
doc.wasGeneratedBy(enes_xarray,read_zarr)

doc.used(subset_zarr, enes_xarray)
doc.wasGeneratedBy(enes_subset, subset_zarr)
doc.wasDerivedFrom(enes_subset, enes_xarray)

doc.used(plot, enes_subset)
doc.used(plot, lw_df_eu)
doc.wasGeneratedBy(output, plot)

### Other relationships

In [ ]:
doc.alternateOf(lw_record, gbif)

### Show Provenance Notation

In [ ]:
print(doc.get_provn())

### Visualization

In [ ]:
from prov.dot import prov_to_dot
figure = prov_to_dot(doc,direction="TB")
figure.set_dpi(120)

In [ ]:
figure.write_png('./enes_lifewatch_provenance.png')

In [ ]:
from IPython.display import Image
Image("./enes_lifewatch_provenance.png")

In [ ]:
doc.serialize('./enes_lifewatch_provenance.json')

### Upload the provenance document to the ENES Provenance service

This requires obtaining a token from https://aai-demo.egi.eu/token to be used in the upload request.

Then, the provenance service will make use of the **Token Introspection Process** for managing authentication and authorization. Specifically:
1. The API extracts the Bearer token from the Authorization header
2. It sends the token to the introspection endpoint (https://enes-proxy.pilot.eosc-beyond.eu/auth/realms/enes/protocol/openid-connect/token/introspect)
3. It checks the introspection response to ensure that the token is valid and includes the required entitlements.

In [ ]:
token = "..."

In [ ]:
yprov_endpoint = "http://yprov.disi.unitn.it:8004"

In [ ]:
from pathlib import Path
import json
import requests

base_url = yprov_endpoint.rstrip("/")
file_path = Path("./enes_lifewatch_US.json")

# Validate the source file before uploading.
original = file_path.read_bytes()
if not original:
    raise ValueError("The file is empty.")
expected_document = json.loads(original)

headers = {"Authorization": f"Bearer {token}"}

metadata = {
    "title": "ENES/LifeWatch User Story",
    "author": "Fabrizio Antonio",
}
    
with file_path.open("rb") as f:
    response = requests.post(
        f"{base_url}/documents",
        headers=headers,
        files={
            "document_file": (
                file_path.name,
                f,
                "application/json",
            )
        },
        params={
            "document_metadata": json.dumps(metadata)
        },
        timeout=120,
    )
    
print("Upload:", response.status_code, response.text)
response.raise_for_status()
record = response.json()

### Attach artifact

In [ ]:
import os, requests
file_path = "./temperature_and_species_occurrences_2000.png"
filename = os.path.basename(file_path)
headers = {"Authorization": f"Bearer {token}"}

r = requests.post(f"{yprov_endpoint}/artifacts/upload/url", params={"filename": filename}, headers=headers)
r.raise_for_status()
upload_info = r.json()
upload_url = upload_info["upload_url"]

with open(file_path, "rb") as f:
    files = {"document_file": (filename, f, "application/octet-stream")}
    r2 = requests.put(upload_url, files=files, headers=headers)
        
r2.raise_for_status()
print(r2.json())